# Phase 1 — PyTorch Tensors

The **tensor** is PyTorch's core data structure — everything in deep learning
(inputs, weights, activations, gradients) is a tensor.

| Structure | Library | Role |
|---|---|---|
| `list` | Python | general-purpose container, no math built in |
| `ndarray` | NumPy | fast numerical arrays, CPU only |
| `Tensor` | PyTorch | like a NumPy array, **plus** GPU support and automatic differentiation (Phase 2) |

## Roadmap for this notebook
```
Creating tensors -> attributes -> dtype & device -> indexing/slicing
    -> operations & broadcasting -> reshaping -> NumPy bridge
    -> mini preview: why we'll need autograd next
```


## Section 1 — Creating Tensors

Tensors can be built directly from Python data, or generated with factory
functions (the way you'll usually create weights, biases, and dummy data).

In [1]:
import torch

torch.manual_seed(42)  # makes every "random" tensor below reproducible across runs

# ── FROM PYTHON DATA ──────────────────────────────────────────────────────────
# torch.tensor() copies data into a new tensor (like np.array())
t_list   = torch.tensor([1, 2, 3, 4, 5])          # from a list -> 1D tensor (vector)
t_float  = torch.tensor([1.0, 2.0, 3.0])          # dtype inferred as float32
t_nested = torch.tensor([[1, 2, 3], [4, 5, 6]])   # from nested list -> 2D tensor (matrix)

print('=== From Python Data ===')
print(f'1D tensor : {t_list}')
print(f'2D tensor :\n{t_nested}')

# ── FACTORY FUNCTIONS — the standard way to create tensors for models ────────
zeros  = torch.zeros(3, 3)          # all zeros, shape (3, 3)
ones   = torch.ones(2, 4)           # all ones, shape (2, 4)
rand   = torch.rand(5, 3)           # uniform random in [0, 1), shape (5, 3)
randn  = torch.randn(2, 2)          # standard normal (mean=0, std=1) -- used to init weights
arange = torch.arange(0, 10, 2)     # like Python range(): start, stop, step
linspc = torch.linspace(0, 1, 5)    # 5 evenly spaced values between 0 and 1
eye    = torch.eye(3)               # 3x3 identity matrix

print('\n=== Factory Functions ===')
print(f'zeros(3,3)     :\n{zeros}')
print(f'ones(2,4)      :\n{ones}')
print(f'rand(5,3)      :\n{rand}')
print(f'randn(2,2)     :\n{randn}')
print(f'arange(0,10,2) : {arange}')
print(f'linspace(0,1,5): {linspc}')
print(f'eye(3)         :\n{eye}')

# ── TENSOR ATTRIBUTES — always check these when debugging shape errors ───────
x = torch.rand(4, 3)
print('\n=== Tensor Attributes ===')
print(f'shape  : {x.shape}      (rows, cols) -- same as x.size()')
print(f'ndim   : {x.ndim}       number of dimensions')
print(f'dtype  : {x.dtype}   element data type')
print(f'device : {x.device}       where the data lives (cpu/cuda/mps)')
print(f'numel  : {x.numel()}      total number of elements')

=== From Python Data ===
1D tensor : tensor([1, 2, 3, 4, 5])
2D tensor :
tensor([[1, 2, 3],
        [4, 5, 6]])

=== Factory Functions ===
zeros(3,3)     :
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
ones(2,4)      :
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])
rand(5,3)      :
tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408],
        [0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411]])
randn(2,2)     :
tensor([[-1.2284,  0.5294],
        [ 1.2211,  0.1511]])
arange(0,10,2) : tensor([0, 2, 4, 6, 8])
linspace(0,1,5): tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
eye(3)         :
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

=== Tensor Attributes ===
shape  : torch.Size([4, 3])      (rows, cols) -- same as x.size()
ndim   : 2       number of dimensions
dtype  : torch.float32   element data type
device : cpu       where the data lives (cpu/cuda/mps)
numel  : 12      total nu

## Section 2 — Data Types (dtype) & Device

Two attributes determine how a tensor behaves: its **dtype** (what kind of
numbers it stores) and its **device** (where the data physically lives —
CPU RAM, or a GPU's memory).

In [2]:
# ── DATA TYPES (dtype) ────────────────────────────────────────────────────────
int_t    = torch.tensor([1, 2, 3])                        # default int64 (torch.long)
float_t  = torch.tensor([1.0, 2.0, 3.0])                  # default float32
explicit = torch.tensor([1, 2, 3], dtype=torch.float32)   # force a dtype

print('=== Data Types ===')
print(f'int_t.dtype    : {int_t.dtype}')
print(f'float_t.dtype  : {float_t.dtype}')
print(f'explicit.dtype : {explicit.dtype}')

# Casting between dtypes -- needed constantly (e.g. loss functions expect float)
casted  = int_t.float()               # int64 -> float32
casted2 = float_t.long()              # float32 -> int64 (truncates decimals)
casted3 = float_t.to(torch.float64)   # .to() is the general-purpose way to cast/move
print(f'\nint_t.float()       : {casted} dtype={casted.dtype}')
print(f'float_t.long()      : {casted2} dtype={casted2.dtype}')
print(f'float_t.to(float64) : {casted3} dtype={casted3.dtype}')

# ── DEVICE — where the tensor's data physically lives ────────────────────────
# Training on a GPU (cuda) or Apple Silicon (mps) is much faster than CPU.
# Always write device-agnostic code so your notebook runs anywhere:
device = (
    'cuda' if torch.cuda.is_available()               # NVIDIA GPU
    else 'mps' if torch.backends.mps.is_available()    # Apple Silicon GPU
    else 'cpu'                                          # fallback
)
print(f'\nAvailable device: {device}')

cpu_tensor = torch.rand(2, 2)
moved = cpu_tensor.to(device)   # .to(device) copies data to that device; original is untouched
print(f'cpu_tensor.device : {cpu_tensor.device}')
print(f'moved.device      : {moved.device}')

# NOTE: tensors on different devices CANNOT interact directly.
# cpu_tensor + moved would raise a RuntimeError if device != 'cpu' --
# move both operands to the same device first.

=== Data Types ===
int_t.dtype    : torch.int64
float_t.dtype  : torch.float32
explicit.dtype : torch.float32

int_t.float()       : tensor([1., 2., 3.]) dtype=torch.float32
float_t.long()      : tensor([1, 2, 3]) dtype=torch.int64
float_t.to(float64) : tensor([1., 2., 3.], dtype=torch.float64) dtype=torch.float64

Available device: mps
cpu_tensor.device : cpu
moved.device      : mps:0


## Section 3 — Indexing & Slicing

Tensor indexing works like NumPy: basic indexing, slicing, boolean masks, and
"fancy" (list-based) indexing all apply.

In [3]:
t = torch.arange(1, 13).reshape(3, 4)   # 3x4 tensor: [[1..4], [5..8], [9..12]]
print('=== Base Tensor ===')
print(t)

# ── BASIC INDEXING ────────────────────────────────────────────────────────────
print('\n=== Indexing ===')
print(f't[0]    = {t[0]}          -> first row')
print(f't[0, 2] = {t[0, 2].item()}                -> row 0, col 2 (scalar)')
print(f't[-1]   = {t[-1]}       -> last row')

# ── SLICING ────────────────────────────────────────────────────────────────────
print('\n=== Slicing ===')
print(f't[:, 0]     = {t[:, 0]}       -> all rows, column 0')
print(f't[0:2, 1:3] =\n{t[0:2, 1:3]}   -> rows 0-1, cols 1-2')
print(f't[:, -1]    = {t[:, -1]}       -> last column')

# ── BOOLEAN MASKING — filter elements by condition ────────────────────────────
mask = t > 6
print('\n=== Boolean Masking ===')
print(f'mask (t > 6):\n{mask}')
print(f't[t > 6]     : {t[mask]}   -> flat tensor of matching elements')

# ── FANCY INDEXING — select specific rows/cols by index list ─────────────────
print('\n=== Fancy Indexing ===')
print(f't[[0, 2]]     : rows 0 and 2 ->\n{t[[0, 2]]}')
print(f't[:, [0, 3]]  : cols 0 and 3 ->\n{t[:, [0, 3]]}')

# ── .item() — pull a single-element tensor out as a plain Python number ─────
scalar = t[1, 1]
print(f'\nt[1,1] as tensor : {scalar}')
print(f't[1,1].item()    : {scalar.item()}  (type: {type(scalar.item()).__name__})')

=== Base Tensor ===
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])

=== Indexing ===
t[0]    = tensor([1, 2, 3, 4])          -> first row
t[0, 2] = 3                -> row 0, col 2 (scalar)
t[-1]   = tensor([ 9, 10, 11, 12])       -> last row

=== Slicing ===
t[:, 0]     = tensor([1, 5, 9])       -> all rows, column 0
t[0:2, 1:3] =
tensor([[2, 3],
        [6, 7]])   -> rows 0-1, cols 1-2
t[:, -1]    = tensor([ 4,  8, 12])       -> last column

=== Boolean Masking ===
mask (t > 6):
tensor([[False, False, False, False],
        [False, False,  True,  True],
        [ True,  True,  True,  True]])
t[t > 6]     : tensor([ 7,  8,  9, 10, 11, 12])   -> flat tensor of matching elements

=== Fancy Indexing ===
t[[0, 2]]     : rows 0 and 2 ->
tensor([[ 1,  2,  3,  4],
        [ 9, 10, 11, 12]])
t[:, [0, 3]]  : cols 0 and 3 ->
tensor([[ 1,  4],
        [ 5,  8],
        [ 9, 12]])

t[1,1] as tensor : 6
t[1,1].item()    : 6  (type: int)


## Section 4 — Operations & Broadcasting

Arithmetic operators (`+ - * /`) are **element-wise** by default. Matrix
multiplication is a separate operator (`@` / `torch.matmul`). When two
tensors have different (but compatible) shapes, PyTorch **broadcasts** the
smaller one instead of raising an error.

In [4]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# ── ELEMENT-WISE ARITHMETIC ───────────────────────────────────────────────────
print('=== Element-wise Ops ===')
print(f'a + b = {a + b}')
print(f'a - b = {a - b}')
print(f'a * b = {a * b}   <- NOT matrix multiplication, just element-wise')
print(f'a / b = {a / b}')

# ── BROADCASTING — operate on tensors of different shapes ────────────────────
# The smaller tensor is "stretched" (conceptually, not copied) to match the
# larger one's shape. Rule: trailing dimensions must match, or be 1.
m   = torch.ones(3, 3)
row = torch.tensor([1.0, 2.0, 3.0])   # shape (3,)
print('\n=== Broadcasting: (3,3) + (3,) ===')
print(f'm:\n{m}')
print(f'row: {row}')
print(f'm + row:\n{m + row}   <- row is added to every row of m')

# ── MATRIX MULTIPLICATION ─────────────────────────────────────────────────────
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print('\n=== Matrix Multiplication ===')
print(f'A @ B (same as torch.matmul):\n{A @ B}')
print(f'A * B (element-wise, NOT matmul):\n{A * B}')

# ── REDUCTION OPERATIONS ──────────────────────────────────────────────────────
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print('\n=== Reductions ===')
print(f'x.sum()      = {x.sum()}          total')
print(f'x.sum(dim=0) = {x.sum(dim=0)}    column-wise sum (collapses rows)')
print(f'x.sum(dim=1) = {x.sum(dim=1)}    row-wise sum (collapses columns)')
print(f'x.mean()     = {x.mean()}')
print(f'x.max()      = {x.max()}')
print(f'x.argmax()   = {x.argmax()}       index of max in the FLATTENED tensor')

# ── IN-PLACE OPERATIONS ───────────────────────────────────────────────────────
# Methods ending in "_" modify the tensor in place instead of returning a new
# one. Saves memory, but in-place ops on tensors that require_grad can break
# autograd's computation graph -- Phase 2 covers this in detail.
c = torch.tensor([1.0, 2.0, 3.0])
print(f'\nBefore add_: {c}')
c.add_(10)   # c = c + 10, done in-place
print(f'After c.add_(10): {c}')

=== Element-wise Ops ===
a + b = tensor([5., 7., 9.])
a - b = tensor([-3., -3., -3.])
a * b = tensor([ 4., 10., 18.])   <- NOT matrix multiplication, just element-wise
a / b = tensor([0.2500, 0.4000, 0.5000])

=== Broadcasting: (3,3) + (3,) ===
m:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
row: tensor([1., 2., 3.])
m + row:
tensor([[2., 3., 4.],
        [2., 3., 4.],
        [2., 3., 4.]])   <- row is added to every row of m

=== Matrix Multiplication ===
A @ B (same as torch.matmul):
tensor([[19., 22.],
        [43., 50.]])
A * B (element-wise, NOT matmul):
tensor([[ 5., 12.],
        [21., 32.]])

=== Reductions ===
x.sum()      = 21.0          total
x.sum(dim=0) = tensor([5., 7., 9.])    column-wise sum (collapses rows)
x.sum(dim=1) = tensor([ 6., 15.])    row-wise sum (collapses columns)
x.mean()     = 3.5
x.max()      = 6.0
x.argmax()   = 5       index of max in the FLATTENED tensor

Before add_: tensor([1., 2., 3.])
After c.add_(10): tensor([11., 12., 13.]

## Section 5 — Reshaping Tensors

Reshaping never changes the underlying data — only how it's "viewed" in terms
of dimensions. This matters a lot in practice: model layers require inputs
of a specific shape (e.g. `(batch_size, channels, height, width)`).

In [5]:
t = torch.arange(1, 13)   # 1D tensor, 12 elements
print(f'Original: {t}  shape={t.shape}')

# ── VIEW vs RESHAPE ────────────────────────────────────────────────────────
# .view() requires the tensor to be "contiguous" in memory and shares the
# same underlying data (no copy). .reshape() does the same when possible,
# but silently copies if the data isn't contiguous. Prefer .reshape() unless
# you specifically need the no-copy guarantee.
v = t.view(3, 4)
r = t.reshape(3, 4)
print(f'\nview(3,4):\n{v}')
print(f'reshape(3,4):\n{r}')

# Both share memory with the original -- modifying one affects the other
v[0, 0] = 999
print(f'\nAfter v[0,0]=999, original t: {t}   <- t changed too! (shared memory)')
t[0] = 1  # restore for later cells

# ── TRANSPOSE & PERMUTE ───────────────────────────────────────────────────────
mat = torch.arange(1, 7).reshape(2, 3)
print(f'\nmat (2,3):\n{mat}')
print(f'mat.transpose(0,1) (3,2):\n{mat.transpose(0, 1)}   <- swaps dims 0 and 1')

cube = torch.rand(2, 3, 4)            # shape (2, 3, 4)
permuted = cube.permute(2, 0, 1)      # reorder dims -> shape (4, 2, 3)
print(f'\ncube.shape: {cube.shape}')
print(f'cube.permute(2,0,1).shape: {permuted.shape}   <- reorders ALL dims at once')

# ── SQUEEZE / UNSQUEEZE ───────────────────────────────────────────────────────
# Common need: add/remove a size-1 dimension (e.g. a batch dim for one sample)
s = torch.rand(3, 1, 4)
print(f'\ns.shape: {s.shape}')
print(f's.squeeze().shape  : {s.squeeze().shape}      <- removes ALL size-1 dims')
print(f's.squeeze(1).shape : {s.squeeze(1).shape}      <- removes only dim 1')

single = torch.rand(4)
print(f'\nsingle.shape: {single.shape}')
print(f'single.unsqueeze(0).shape: {single.unsqueeze(0).shape}  <- adds a batch dim at the front')

# ── FLATTEN ────────────────────────────────────────────────────────────────────
img = torch.rand(2, 3, 4)   # e.g. (channels, height, width)
print(f'\nimg.shape: {img.shape}')
print(f'img.flatten().shape             : {img.flatten().shape}   <- collapses to 1D')
print(f'img.flatten(start_dim=1).shape  : {img.flatten(start_dim=1).shape}  <- keeps dim 0, flattens the rest')

Original: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])  shape=torch.Size([12])

view(3,4):
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
reshape(3,4):
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])

After v[0,0]=999, original t: tensor([999,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12])   <- t changed too! (shared memory)

mat (2,3):
tensor([[1, 2, 3],
        [4, 5, 6]])
mat.transpose(0,1) (3,2):
tensor([[1, 4],
        [2, 5],
        [3, 6]])   <- swaps dims 0 and 1

cube.shape: torch.Size([2, 3, 4])
cube.permute(2,0,1).shape: torch.Size([4, 2, 3])   <- reorders ALL dims at once

s.shape: torch.Size([3, 1, 4])
s.squeeze().shape  : torch.Size([3, 4])      <- removes ALL size-1 dims
s.squeeze(1).shape : torch.Size([3, 4])      <- removes only dim 1

single.shape: torch.Size([4])
single.unsqueeze(0).shape: torch.Size([1, 4])  <- adds a batch dim at the front

img.shape: torch.Size([2, 3, 4])
img.flat

## Section 6 — The NumPy Bridge

PyTorch tensors and NumPy arrays can convert to each other almost for free
— **they share the same memory**, so no data is copied. This is fast, but
it's a common source of subtle bugs if you're not expecting it.

In [6]:
import numpy as np

# ── NUMPY -> TORCH ────────────────────────────────────────────────────────────
arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(arr)   # SHARES memory with the numpy array (no copy)
print(f'numpy array : {arr}')
print(f'torch tensor: {t_from_np}')

arr[0] = 999.0   # modifying the numpy array...
print(f'\nAfter arr[0]=999: torch tensor is {t_from_np}   <- changed too! (shared memory)')

# ── TORCH -> NUMPY ────────────────────────────────────────────────────────────
t2 = torch.tensor([4.0, 5.0, 6.0])
np_from_t = t2.numpy()   # also shares memory (CPU tensors only)
print(f'\ntensor: {t2}')
print(f'.numpy(): {np_from_t}   type={type(np_from_t).__name__}')

# NOTE: .numpy() only works on CPU tensors. A tensor on 'cuda'/'mps' must be
# moved with .cpu() first: gpu_tensor.cpu().numpy()

# ── AVOIDING SHARED MEMORY — use .clone() / np.copy() for an independent copy ─
t_copy = torch.from_numpy(arr).clone()
arr[0] = -1.0
print(f'\nAfter modifying arr again, t_copy (cloned) is unaffected: {t_copy}')

numpy array : [1. 2. 3.]
torch tensor: tensor([1., 2., 3.], dtype=torch.float64)

After arr[0]=999: torch tensor is tensor([999.,   2.,   3.], dtype=torch.float64)   <- changed too! (shared memory)

tensor: tensor([4., 5., 6.])
.numpy(): [4. 5. 6.]   type=ndarray

After modifying arr again, t_copy (cloned) is unaffected: tensor([999.,   2.,   3.], dtype=torch.float64)


## Section 7 — Preview: Why We'll Need Autograd Next

Here's a tiny "model": `y_pred = w * x + b`. We can already compute a loss
that tells us *how wrong* the prediction is — but not *which direction* to
nudge `w` and `b` to make it better.

In [7]:
# This mirrors real training: we have inputs x, true outputs y, and
# parameters w (weight) and b (bias) that we want to learn.
x = torch.tensor([1.0, 2.0, 3.0, 4.0])   # inputs
y = torch.tensor([3.0, 5.0, 7.0, 9.0])   # true outputs (the real relationship is y = 2x + 1)

# A wrong first guess for the parameters
w = torch.tensor(0.0)
b = torch.tensor(0.0)

# ── FORWARD PASS — the model's prediction ────────────────────────────────────
y_pred = w * x + b

# ── LOSS — how wrong is the prediction? (Mean Squared Error) ────────────────
loss = ((y_pred - y) ** 2).mean()

print(f'inputs (x)         : {x}')
print(f'true values (y)    : {y}')
print(f'predicted (y_pred) : {y_pred}')
print(f'loss (MSE)         : {loss.item():.4f}')

# We now know HOW WRONG we are, but not WHICH DIRECTION to move w and b to
# reduce the loss. Working that out by hand (calculus, chain rule) for every
# parameter in a real network is impractical. That's the exact problem
# Autograd (Phase 2) solves: automatically computing these gradients for us.

inputs (x)         : tensor([1., 2., 3., 4.])
true values (y)    : tensor([3., 5., 7., 9.])
predicted (y_pred) : tensor([0., 0., 0., 0.])
loss (MSE)         : 41.0000


## Phase 1 Summary

```
Creating Tensors:
  torch.tensor([...])         -> from Python data
  torch.zeros/ones/rand/randn(shape)
  torch.arange(start,stop,step) / torch.linspace(start,stop,n)
  torch.eye(n)                -> identity matrix

Attributes:
  t.shape / t.size()   t.ndim   t.dtype   t.device   t.numel()

Dtype & Device:
  t.float() / t.long() / t.to(dtype=...)
  t.to(device)          -> move between cpu / cuda / mps
  device-agnostic pattern:
    device = "cuda" if torch.cuda.is_available() \
             else "mps" if torch.backends.mps.is_available() else "cpu"

Indexing & Slicing:
  t[i], t[i, j], t[:, j], t[i:j, k:l]
  t[t > val]              -> boolean mask
  t[[i, j]]                -> fancy indexing
  t.item()                 -> single-element tensor -> Python number

Operations:
  + - * /                  -> element-wise (broadcasting applies)
  a @ b / torch.matmul     -> matrix multiplication
  t.sum(dim=), t.mean(), t.max(), t.argmax()
  t.add_(x)                -> in-place (trailing underscore)

Reshaping:
  t.view(shape) / t.reshape(shape)   -> same data, new shape
  t.transpose(d0, d1) / t.permute(*dims)
  t.squeeze() / t.unsqueeze(dim)     -> remove/add size-1 dims
  t.flatten(start_dim=)

NumPy Bridge:
  torch.from_numpy(arr)  /  t.numpy()   -> share memory (CPU only!)
  .clone()  /  np.copy()                -> break the shared-memory link

Up Next (Phase 2 — Autograd):
  requires_grad=True, loss.backward(), t.grad, torch.no_grad()
```